# 16. Residual Stream and Activation Analysis（spec §11）

## 学習目標
- forward hook で記録した各層の活性 RMS を層×チェックポイントで俯瞰する
- 各ブロックが residual stream をどれだけ書き換えるか $r_l=\|\Delta h_l\|/\|h_l\|$ を測る
- 活性の異常（爆発/消失/外れ値チャネル）検出の観点を知る

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.visualization import curves
records = read_jsonl(M2_RUN / "metrics.jsonl")
curves.activation_rms_heatmap(records).show()

In [3]:
# branch別 residual 書き込み比 r_l を最終モデルの forward trace から
from jp_llm_lab.reporting.analysis_artifacts import _load_model, _load_tokenizer
from jp_llm_lab.instrumentation.activation_stats import residual_update_ratios
model = _load_model(sorted(M2_RUN.glob("checkpoints/*100pct*.pt"))[0])
tok = _load_tokenizer(M2_RUN)
ids = torch.tensor([tok.encode("日本の首都は東京です。私はその街に住んでいます。")[:64]])
trace = model.trace_forward(ids)
ratios = residual_update_ratios(trace, n_layers=model.cfg.n_layers)
names = list(ratios)
fig = go.Figure(go.Bar(x=names, y=[ratios[n] for n in names],
                       marker_color=["#2ca02c" if "attn" in n else "#ff7f0e" for n in names]))
fig.update_layout(title="branch別 residual 書き込み比 r_l=‖Δh‖/‖h‖（緑=attn, 橙=mlp）",
                  yaxis_title="r_l", template="plotly_white", height=360)
fig.show()

In [4]:
# 各観測点の活性統計（最終eval）
evals = [r for r in records if r["type"]=="eval"]
last = evals[-1]["activation_stats"]
rows = [(k, v["rms"], v["kurtosis"], v["absmax"], v["outlier_frac"]) for k,v in last.items()]
df = pd.DataFrame(rows, columns=["point","rms","kurtosis","absmax","outlier_frac"])
print("kurtosis>3 は重い裾（外れ値チャネルの兆候）。NaN/Inf:", evals[-1]["nonfinite"] or "なし")
df.round(3)

kurtosis>3 は重い裾（外れ値チャネルの兆候）。NaN/Inf: なし


,point,rms,kurtosis,absmax,outlier_frac
0,tok_emb,0.029,2.880,0.122,0.0
1,blocks.0.attn,0.015,5.253,0.229,0.0
2,blocks.0.mlp,0.626,3.222,4.034,0.0
3,blocks.0.resid,0.636,3.210,4.055,0.0
4,blocks.1.attn,0.092,3.902,0.701,0.0
5,blocks.1.mlp,0.211,3.296,1.581,0.0
6,blocks.1.resid,0.622,3.122,3.929,0.0
7,blocks.2.attn,0.070,5.020,0.718,0.0
8,blocks.2.mlp,0.148,3.424,1.268,0.0
9,blocks.2.resid,0.608,3.257,3.875,0.0


## Observation / Interpretation / Caveat
- **Observation**: residual RMS は層とともに増加（pre-LN の蓄積）。$r_l$ は MLP branch が attention branch より大きい層が多い。NaN/Inf なし。一部の点で kurtosis>3（裾が重い）。
- **Interpretation**: 各ブロックは residual stream を「置換」ではなく「小さく加算」しており、深い層ほど絶対値が育つ。重い裾は特定チャネルに大きな値が乗る既知の現象（outlier feature）で、量子化などで問題になりうる。
- **Caveat**: $r_l$ は1文・1シードの値。RMS は分布の1統計にすぎず、kurtosis/absmax と併読が必要。

**次へ**: [17_gradient_analysis](17_gradient_analysis.ipynb)